# TabR POC-ABS Native Event-Level Training and External Datatest Evaluation

Notebook utama untuk training model POC-ABS dengan **1 row = 1 event**. Source Excel tetap frame-level, tetapi saat export dataset TabR frame dalam event digabung, diurutkan berdasarkan `frame`, lalu diekspansi menjadi fitur temporal (`*_t001`, `*_t002`, dst).

Split qholdout/qwalkforward tetap event-safe, kalibrasi threshold dilakukan langsung pada prediksi event validation, dan external datatest juga diprediksi langsung di level event. Artifact lama tidak ditimpa karena dataset/experiment memakai suffix versi baru.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
TABR_ROOT = ROOT / 'third_party' / 'tabular-dl-tabr-official'
os.environ['PROJECT_DIR'] = str(TABR_ROOT)
if str(TABR_ROOT) not in sys.path:
    sys.path.append(str(TABR_ROOT))

import bin.go
from scripts.export_tabr_pocabs_event import EXPERIMENTS, export_all
from scripts.evaluate_tabr_pocabs_frame import OUT_ROOT, evaluate_all

N_EVAL_SEEDS = 15
N_ENSEMBLES = 0
ENSEMBLE_SIZE = 5
display(Markdown(f'ROOT: `{ROOT}`  \nTABR_ROOT: `{TABR_ROOT}`'))

display(Markdown('Existing-model subject-bias-reduced evaluation: event threshold and subject threshold are calibrated separately; subject aggregation tries mean/median/trimmed_mean/p90/top3_mean/proportion/count.'))


ROOT: `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st`  
TABR_ROOT: `/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/third_party/tabular-dl-tabr-official`

Existing-model subject-bias-reduced evaluation: event threshold and subject threshold are calibrated separately; subject aggregation tries mean/median/trimmed_mean/p90/top3_mean/proportion/count.

## 1. Export new official TabR native event-level datasets

This creates new versioned dataset and experiment directories. The source Excel remains frame-level, but this export reshapes it into native event-level rows before writing TabR arrays. If directories already exist, `overwrite=False` prevents accidental overwrite.


In [ ]:
export_results = export_all(overwrite=False)
for item in export_results:
    display(Markdown(f"### `{item['exp_dir']}`"))
    display(pd.DataFrame(item['splits']))
    print('feature_count:', item['feature_count'])
    print('tuning_toml:', item['tuning_toml'])


## 2. Train new TabR models

Uses the official TabR `bin.go.main` runner/library, not a shell script. `force=False` keeps old runs safe; `continue_=True` allows resume.


In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

In [ ]:
for cfg in EXPERIMENTS:
    tuning_toml = TABR_ROOT / 'exp' / 'tabr' / cfg['exp_name'] / '0-tuning.toml'
    display(Markdown(f"## Training `{cfg['exp_name']}`"))
    bin.go.main(
        tuning_toml,
        n_seeds=N_EVAL_SEEDS,
        n_ensembles=N_ENSEMBLES,
        ensemble_size=ENSEMBLE_SIZE,
        continue_=True,
        force=False,
    )


## 3. Native event-level validation calibration and external datatest evaluation

For each experiment, this evaluates all seeds on validation event-level metrics, chooses the best seed + threshold, then evaluates external datatest. Event predictions are produced directly from event rows; subject-level metrics still aggregate event probabilities.


In [ ]:
outputs = evaluate_all(return_predictions=False)
display(Markdown(f'Results saved to: `{OUT_ROOT}`'))


In [ ]:
csv_files = pd.DataFrame({'file': [p.name for p in sorted(OUT_ROOT.glob('*.csv'))], 'path': [str(p) for p in sorted(OUT_ROOT.glob('*.csv'))], 'size_mb': [p.stat().st_size / 1024 / 1024 for p in sorted(OUT_ROOT.glob('*.csv'))]})
display(csv_files)


In [ ]:
for name in ['calibration', 'event_summary', 'subject_summary', 'detail_files']:
    display(Markdown(f'## {name}'))
    display(outputs[name].head(20))


## 4. Best external event-level results


In [ ]:
event_summary = outputs['event_summary'].copy()
display(event_summary.sort_values(['scenario', 'balanced_accuracy', 'macro_f1'], ascending=[True, False, False]).reset_index(drop=True))

subject_summary = outputs['subject_summary'].copy()
display(Markdown('## Subject-derived summary'))
display(subject_summary.sort_values(['scenario', 'balanced_accuracy', 'macro_f1'], ascending=[True, False, False]).reset_index(drop=True))
